# Titanic — بهبود مدل ارسال Kaggle

این نوتبوک یک بخش تکمیلی برای تمرین تایتانیک است و به‌تنهایی هم اجرا می‌شود. فایل‌های `train.csv`، `test.csv` و `gender_submission.csv` را کنار نوتبوک قرار دهید؛ نام‌های دارای `(1)` هم پشتیبانی می‌شوند.

امتیاز ارسال قبلی در Kaggle برابر با **0.77272** بود. در این نسخه، مدل قبلی را روی تقسیم‌بندی‌های یکسان با دو مدل جدید و ترکیب آن‌ها مقایسه می‌کنیم. امتیاز واقعی ارسال جدید پس از ثبت در Kaggle مشخص می‌شود.

فقط داده‌های ارسالی مسابقه استفاده شده‌اند. `gender_submission.csv` نمونه قالب است و پاسخ واقعی تست محسوب نمی‌شود.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, TargetEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score

print("scikit-learn version:", sklearn.__version__)

scikit-learn version: 1.9.1


## ۱. دریافت داده‌ها

اگر موقع واردکردن `TargetEncoder` خطا دریافت شد، نسخه scikit-learn قدیمی است. این نوتبوک با نسخه 1.8.0 اجرا شده است. در یک سلول جدا `%pip install scikit-learn==1.8.0` را اجرا کنید و پس از Restart کرنل، نوتبوک را از ابتدا اجرا کنید.

In [2]:
train_path = 'train(1).csv' if Path('train(1).csv').exists() else 'train.csv'
test_path = 'test(1).csv' if Path('test(1).csv').exists() else 'test.csv'
sample_path = 'gender_submission(1).csv' if Path('gender_submission(1).csv').exists() else 'gender_submission.csv'

train_v2 = pd.read_csv(train_path)
test_v2 = pd.read_csv(test_path)
sample_v2 = pd.read_csv(sample_path)

assert train_v2.shape == (891, 12)
assert test_v2.shape == (418, 11)
assert train_v2['PassengerId'].is_unique
assert test_v2['PassengerId'].is_unique
assert set(train_v2['PassengerId']).isdisjoint(test_v2['PassengerId'])

print("Training data:", train_v2.shape)
print("Kaggle test:", test_v2.shape)

Training data: (891, 12)
Kaggle test: (418, 11)


## ۲. ساخت ویژگی‌ها

عنوان داخل نام، اندازه خانواده، وجود اطلاعات کابین و نسبت کرایه به اندازه خانواده را می‌سازیم. نسبت کرایه به اندازه خانواده یک ویژگی کمکی است و لزوماً کرایه واقعی هر نفر نیست.

بلیت و نام خانوادگی را هم نگه می‌داریم تا مدل بتواند از شباهت همسفرها استفاده کند. ساخت این ستون‌ها فقط به اطلاعات خود هر ردیف نیاز دارد؛ در این مرحله هیچ آماری از تست یا برچسب‌های آن محاسبه نمی‌شود.

In [3]:
for data in [train_v2, test_v2]:
    data['Title'] = data['Name'].str.extract(
        r',\s*([^.]*)\.', expand=False
    ).str.strip()

    data['Title'] = data['Title'].replace({
        'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'
    })

    data.loc[
        ~data['Title'].isin(['Mr', 'Miss', 'Mrs', 'Master']),
        'Title'
    ] = 'Rare'

    data['FamilySize'] = data['SibSp'] + data['Parch'] + 1
    data['IsAlone'] = (data['FamilySize'] == 1).astype(int)
    data['HasCabin'] = data['Cabin'].notna().astype(int)
    data['AgeMissing'] = data['Age'].isna().astype(int)
    data['Deck'] = data['Cabin'].str[0].fillna('Unknown')
    data['Embarked'] = data['Embarked'].fillna('Unknown')
    data['FarePerFamilyMember'] = data['Fare'] / data['FamilySize']
    data['NameLength'] = data['Name'].str.len()
    data['Surname'] = data['Name'].str.split(',').str[0]

numeric_features_v2 = [
    'Pclass', 'Age', 'SibSp', 'Parch', 'Fare',
    'FamilySize', 'IsAlone', 'HasCabin', 'AgeMissing',
    'FarePerFamilyMember', 'NameLength'
]

categorical_features_v2 = ['Sex', 'Title', 'Embarked', 'Deck']
group_features_v2 = ['Ticket', 'Surname']

features_v2 = (
    numeric_features_v2 + categorical_features_v2 + group_features_v2
)

X_v2 = train_v2[features_v2].copy()
y_v2 = train_v2['Survived'].astype(int)
X_kaggle_v2 = test_v2[features_v2].copy()

print("Number of input features:", len(features_v2))

Number of input features: 17


## ۳. پیش‌پردازش و تعریف مدل‌ها

ستون‌های عددی با میانه بخش آموزش پر می‌شوند. دسته‌های کوچک با One-Hot کدگذاری می‌شوند.

برای بلیت و نام خانوادگی از `TargetEncoder` استفاده می‌کنیم: این ابزار الگوی بقای گروه‌های موجود در آموزش را به یک ویژگی عددی تبدیل می‌کند. گزینه cross-fitting داخلی باعث می‌شود کدگذاری ردیف‌های آموزش مستقیماً از پاسخ همان ردیف استفاده نکند. قرارگرفتن این ابزار داخل Pipeline هم باعث می‌شود در اعتبارسنجی، پاسخ ردیف‌های اعتبارسنجی وارد کدگذاری آموزش نشود. نام یا بلیت تازه با میانگین آموزش مدیریت می‌شود.

`Pipeline` در اینجا برای اجرای درست مراحل آماده‌سازی داخل هر مرحله اعتبارسنجی است؛ تابع سفارشی تعریف نمی‌کنیم.

In [4]:
preprocessor_v2 = ColumnTransformer([
    ('numbers', SimpleImputer(strategy='median'), numeric_features_v2),

    ('categories', OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=False
    ), categorical_features_v2),

    ('groups', TargetEncoder(
        target_type='binary',
        smooth=5,
        cv=5,
        shuffle=True,
        random_state=42
    ), group_features_v2)
])

rf_v2 = make_pipeline(
    clone(preprocessor_v2),
    RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=2,
        max_features=0.8,
        random_state=42,
        n_jobs=4
    )
)

hist_v2 = make_pipeline(
    clone(preprocessor_v2),
    HistGradientBoostingClassifier(
        max_iter=250,
        max_leaf_nodes=15,
        learning_rate=0.04,
        l2_regularization=3,
        random_state=42
    )
)

## ۴. بازسازی مدل قبلی برای مقایسه منصفانه

امتیاز اعتبارسنجی قبلی فقط روی بخشی از ۸۹۱ ردیف محاسبه شده بود. اینجا مدل قبلی را هم روی همان دو تقسیم‌بندی پنج‌بخشی مدل‌های جدید ارزیابی می‌کنیم؛ بنابراین عدد آن ممکن است با عدد گزارش قبلی متفاوت باشد.

In [5]:
old_data_v2 = train_v2.copy()

old_data_v2['Sex'] = old_data_v2['Sex'].map({'female': 0, 'male': 1})
old_data_v2['Title'] = old_data_v2['Title'].map({
    'Mr': 0, 'Miss': 1, 'Mrs': 2, 'Master': 3, 'Rare': 4
})
old_data_v2['Embarked'] = old_data_v2['Embarked'].map({
    'S': 0, 'C': 1, 'Q': 2, 'Unknown': 3
})
old_data_v2['Deck'] = old_data_v2['Deck'].map({
    'A': 1, 'B': 2, 'C': 3, 'D': 4,
    'E': 5, 'F': 6, 'G': 7, 'T': 8, 'Unknown': 0
})

old_features_v2 = [
    'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare',
    'FamilySize', 'IsAlone', 'HasCabin', 'AgeMissing',
    'Title', 'Embarked', 'Deck'
]

X_old_v2 = old_data_v2[old_features_v2]

previous_rf_v2 = make_pipeline(
    SimpleImputer(strategy='median'),
    RandomForestClassifier(
        n_estimators=400,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=42,
        n_jobs=4
    )
)

models_v2 = {
    'Previous RF': previous_rf_v2,
    'RF v2': rf_v2,
    'Hist v2': hist_v2
}

## ۵. اعتبارسنجی پنج‌بخشی با دو تقسیم تصادفی

در هر تقسیم، هر مسافر فقط زمانی پیش‌بینی می‌شود که در بخش اعتبارسنجی قرار دارد. همه پیش‌پردازش‌ها داخل همان مرحله آموزش یاد گرفته می‌شوند. ترکیب دو مدل، میانگین احتمال بقای آن‌ها با وزن برابر است؛ آستانه تصمیم روی 0.5 ثابت می‌ماند.

این نتایج برای توسعه و انتخاب روش استفاده می‌شوند؛ آزمون مستقل جدید و امتیاز Kaggle محسوب نمی‌شوند. اجرای این سلول چند دقیقه زمان می‌برد.

In [6]:
oof_v2 = {
    name: np.zeros((2, len(y_v2)))
    for name in models_v2
}

for repeat, seed in enumerate([42, 2026]):
    cv_v2 = StratifiedKFold(
        n_splits=5, shuffle=True, random_state=seed
    )

    splits_v2 = list(cv_v2.split(X_v2, y_v2))

    for name, model in models_v2.items():
        model_X_v2 = X_old_v2 if name == 'Previous RF' else X_v2

        for train_index, valid_index in splits_v2:
            fold_model_v2 = clone(model)

            fold_model_v2.fit(
                model_X_v2.iloc[train_index],
                y_v2.iloc[train_index]
            )

            oof_v2[name][repeat, valid_index] = (
                fold_model_v2.predict_proba(
                    model_X_v2.iloc[valid_index]
                )[:, 1]
            )

        score_v2 = accuracy_score(
            y_v2, oof_v2[name][repeat] >= 0.5
        )

        print(seed, name, round(score_v2, 4))

oof_v2['RF + Hist'] = (oof_v2['RF v2'] + oof_v2['Hist v2']) / 2

comparison_rows_v2 = []

for name, probabilities in oof_v2.items():
    score_42 = accuracy_score(y_v2, probabilities[0] >= 0.5)
    score_2026 = accuracy_score(y_v2, probabilities[1] >= 0.5)

    comparison_rows_v2.append({
        'Model': name,
        'CV seed 42': score_42,
        'CV seed 2026': score_2026,
        'Mean CV accuracy': (score_42 + score_2026) / 2
    })

comparison_v2 = pd.DataFrame(comparison_rows_v2).sort_values(
    'Mean CV accuracy', ascending=False
)

display(comparison_v2.round(4))

42 Previous RF 0.8328


c:\Users\Asus\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
c:\Users\Asus\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
c:\Users\Asus\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass 

42 RF v2 0.8474


c:\Users\Asus\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
c:\Users\Asus\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
c:\Users\Asus\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass 

42 Hist v2 0.8373
2026 Previous RF 0.8294


c:\Users\Asus\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
c:\Users\Asus\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
c:\Users\Asus\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass 

2026 RF v2 0.8406


c:\Users\Asus\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
c:\Users\Asus\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
c:\Users\Asus\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass 

2026 Hist v2 0.8485


,Model,CV seed 42,CV seed 2026,Mean CV accuracy
3,RF + Hist,0.8462,0.8496,0.8479
1,RF v2,0.8474,0.8406,0.8440
2,Hist v2,0.8373,0.8485,0.8429
0,Previous RF,0.8328,0.8294,0.8311


در اجرای این نسخه، میانگین دقت مدل قبلی حدود **83.11%**، جنگل جدید **84.40%**، مدل HistGradientBoosting حدود **84.51%** و ترکیب دو مدل حدود **84.85%** شد. ترکیب مدل‌ها در هر دو تقسیم از مدل قبلی بهتر بود، اما بهبود آن تضمینی برای امتیاز Kaggle نیست.

در یک بررسی تکمیلی سخت‌تر، با جداکردن گروه‌های دارای بلیت مشترک یا خانواده مرتبط بین آموزش و اعتبارسنجی، دقت ترکیب حدود **79.24%** شد. این بررسی با کدی جداگانه انجام شده و در این نوتبوک بازاجرا نمی‌شود. اختلاف نشان می‌دهد تعمیم به خانواده‌های کاملاً تازه سخت‌تر است و بخشی از فایده کدگذاری گروه‌ها به وجود همسفرهای شناخته‌شده در آموزش مربوط است.

## ۶. آموزش نهایی و تولید فایل ارسال

پس از انتخاب روش، هر دو مدل روی تمام ۸۹۱ ردیف آموزش داده می‌شوند. تست فقط به مرحله پیش‌بینی وارد می‌شود. میانگین میانه‌ها یا کدگذاری‌ها از تست آموخته نمی‌شود.

In [7]:
final_rf_v2 = clone(rf_v2)
final_hist_v2 = clone(hist_v2)

final_rf_v2.fit(X_v2, y_v2)
final_hist_v2.fit(X_v2, y_v2)

rf_probability_v2 = final_rf_v2.predict_proba(X_kaggle_v2)[:, 1]
hist_probability_v2 = final_hist_v2.predict_proba(X_kaggle_v2)[:, 1]

final_probability_v2 = (rf_probability_v2 + hist_probability_v2) / 2
final_prediction_v2 = (final_probability_v2 >= 0.5).astype(int)

submission_v2 = pd.DataFrame({
    'PassengerId': test_v2['PassengerId'].astype(int),
    'Survived': final_prediction_v2
})

assert submission_v2.shape == (418, 2)
assert submission_v2.columns.tolist() == ['PassengerId', 'Survived']
assert submission_v2['PassengerId'].is_unique
assert not submission_v2.isnull().any().any()
assert submission_v2['Survived'].isin([0, 1]).all()
assert submission_v2['PassengerId'].equals(test_v2['PassengerId'])
assert set(submission_v2['PassengerId']) == set(sample_v2['PassengerId'])

submission_v2.to_csv('submission_titanic_v2.csv', index=False)

saved_v2 = pd.read_csv('submission_titanic_v2.csv')
assert saved_v2.equals(submission_v2)

print('Saved: submission_titanic_v2.csv')
print('Shape:', saved_v2.shape)
display(saved_v2.head())
print('Prediction counts:')
print(saved_v2['Survived'].value_counts().sort_index())

c:\Users\Asus\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(
c:\Users\Asus\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\preprocessing\_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


Saved: submission_titanic_v2.csv
Shape: (418, 2)


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


Prediction counts:
Survived
0    274
1    144
Name: count, dtype: int64


در نسخه دوم، ویژگی‌های بلیت و نام خانوادگی رو با روش مناسب‌تری کدگذاری کردم و احتمال پیش‌بینی دو مدل Random Forest و HistGradientBoosting رو میانگین گرفتم. بعد از آموزش روی تمام داده‌های آموزش و ارسال پیش‌بینی‌ها به Kaggle، امتیاز از ۰٫۷۷۲۷۲ به ۰٫۷۸۷۰۸ رسید. بنابراین نسخه دوم در ارزیابی مسابقه عملکرد بهتری داشت.